# Sentinel-1 Level-0 → Focused SLC (Stripmap/S6)


## 1 - Imports and input file

In [ ]:
from time import perf_counter

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib import colors
import sentinel1decoder

import sentinel1_processing.doppler_centroid_estimation as doppler_centroid_estimation
import sentinel1_processing.effective_velocity as effective_velocity
import sentinel1_processing.range_compression as range_compression
import sentinel1_processing.raw_data_correction as raw_data_correction
import sentinel1_processing.slc_assembly as slc_assembly

notebook_started_at = perf_counter()
pd.set_option("display.max_columns", None)


In [ ]:
filepath = "./data/sao_paulo/"
filename = "s1a-s6-raw-s-vv-20251226t214356-20251226t214427-062491-07d496.dat"

inputfile = filepath + filename
l0file = sentinel1decoder.Level0File(inputfile)


## 2 - Metadata and acquisition chunk

In [ ]:
l0file.packet_metadata


In [ ]:
l0file.ephemeris


In [ ]:
selected_chunk = 13

selection = l0file.get_acquisition_chunk_metadata(selected_chunk)
selection


## 3 - Raw signal decoding (DAD §9.1)

Decode the selected acquisition chunk into the complex I/Q matrix used by the Level-1 processing chain.

In [ ]:
radar_data = l0file.get_acquisition_chunk_data(selected_chunk)

len_az_line, raw_len_range_line = radar_data.shape

print("Raw data shape:", radar_data.shape)


In [ ]:
# Quick-look only
plt.figure(figsize=(12, 12))
plt.title("Sentinel-1 Raw I/Q Sensor Output")
plt.imshow(
    np.abs(radar_data[::20, ::20]),
    origin="lower",
    norm=colors.LogNorm()
)
plt.xlabel("Down Range (samples)")
plt.ylabel("Cross Range (samples)")
plt.show()


## 4 - Sampling axes and orbit geometry (DAD §§9.9–9.10)

Build the range/azimuth time bases and the orbit-driven effective-velocity model used by focusing.

In [ ]:
# Radar parameters
c = sentinel1decoder.constants.SPEED_OF_LIGHT_MPS #Tốc độ ánh sáng
wavelength_m = sentinel1decoder.constants.TX_WAVELENGTH_M 

RGDEC = selection["Range Decimation"].iloc[0] 
PRI = selection["PRI"].iloc[0]
rank = selection["Rank"].iloc[0]

TXPSF = selection["Tx Pulse Start Frequency"].iloc[0]
TXPRR = selection["Tx Ramp Rate"].iloc[0]
TXPL = selection["Tx Pulse Length"].iloc[0]

range_sample_freq = sentinel1decoder.utilities.range_dec_to_sample_rate(RGDEC)
range_sample_period = 1.0 / range_sample_freq
az_sample_freq = 1.0 / PRI
az_sample_period = PRI

suppressed_data_time = 320.0 / (8.0 * sentinel1decoder.constants.F_REF)

# Range-time axis of the decoded raw line
range_start_time = selection["SWST"].iloc[0] + suppressed_data_time
raw_fast_time_vec_s = range_start_time + np.arange(raw_len_range_line) / range_sample_freq
raw_slant_range_time_vec_s = rank * PRI + raw_fast_time_vec_s
raw_slant_range_vec_m = raw_slant_range_time_vec_s * c / 2.0

# Azimuth time of each echo packet
packet_azimuth_times_s = (
    selection["Coarse Time"].to_numpy(dtype=float)
    + selection["Fine Time"].to_numpy(dtype=float)
)

# --- PHẦN BỔ SUNG IN TẤT CẢ CÁC BIẾN ---
print("=== HẰNG SỐ & THAM SỐ RADAR HỆ THỐNG ===")
print("Speed of light (c):", c, "m/s")
print("Wavelength:", wavelength_m, "m")
print("Range Decimation (RGDEC):", RGDEC)
print("PRI:", PRI, "s")
print("Rank:", rank)
print("Tx Pulse Start Frequency (TXPSF):", TXPSF, "Hz")
print("Tx Ramp Rate (TXPRR):", TXPRR, "Hz/s")
print("Tx Pulse Length (TXPL):", TXPL, "s")

print("\n=== TẦN SỐ VÀ CHU KỲ LẤY MẪU ===")
print("Range sampling frequency:", range_sample_freq / 1e6, "MHz")
print("Range sampling period:", range_sample_period, "s")
print("PRF (Azimuth sampling frequency):", az_sample_freq, "Hz")
print("Azimuth sampling period:", az_sample_period, "s")
print("Suppressed data time:", suppressed_data_time, "s")

print("\n=== THÔNG SỐ TRỤC RANGE (KHOẢNG CÁCH) ===")
print("Raw range samples (Số lượng mẫu):", raw_len_range_line)
print("Range start time:", range_start_time, "s")

# Đối với mảng dữ liệu (vector), in kích thước và giá trị đầu/cuối để tránh tràn màn hình
print("\n=== DỮ LIỆU VECTOR (MẢNG) ===")
print(f"raw_fast_time_vec_s (Kích thước: {raw_fast_time_vec_s.shape}):")
print(f"  - Bắt đầu: {raw_fast_time_vec_s[0]} s")
print(f"  - Kết thúc: {raw_fast_time_vec_s[-1]} s")

print(f"raw_slant_range_time_vec_s (Kích thước: {raw_slant_range_time_vec_s.shape}):")
print(f"  - Bắt đầu: {raw_slant_range_time_vec_s[0]} s")
print(f"  - Kết thúc: {raw_slant_range_time_vec_s[-1]} s")

print(f"raw_slant_range_vec_m (Kích thước: {raw_slant_range_vec_m.shape}):")
print(f"  - Gần nhất (Near range): {raw_slant_range_vec_m[0] / 1000.0} km")
print(f"  - Xa nhất (Far range): {raw_slant_range_vec_m[-1] / 1000.0} km")

print(f"packet_azimuth_times_s (Kích thước: {packet_azimuth_times_s.shape}):")
print(f"  - Bắt đầu: {packet_azimuth_times_s[0]} s")
print(f"  - Kết thúc: {packet_azimuth_times_s[-1]} s")


In [ ]:
# Precise effective-velocity estimator.
#
# The helper sorts and de-duplicates the L0 PVT orbit epochs, then constructs
# a cubic Hermite orbit interpolator using both ECEF position and velocity.
vr_estimator = effective_velocity.Estimator.from_level0_product(
    l0file,
    wavelength_m
)

print("Unique orbit epochs:", vr_estimator.orbit_times_s.size)
print(
    "Orbit interval:",
    vr_estimator.orbit_times_s[0],
    "to",
    vr_estimator.orbit_times_s[-1],
    "GPS seconds"
)

# Packet Coarse/Fine Time and POD Solution Data Timestamp are in the same
# GPS-time basis in sentinel1decoder. Do not mix these with UTC annotation
# timestamps without applying the GPS-UTC offset.
if (
    packet_azimuth_times_s[0] < vr_estimator.orbit_times_s[0]
    or packet_azimuth_times_s[-1] > vr_estimator.orbit_times_s[-1]
):
    raise ValueError(
        "Echo packet times are outside the available ephemeris interval."
    )

print(
    "Echo interval:",
    packet_azimuth_times_s[0],
    "to",
    packet_azimuth_times_s[-1],
    "GPS seconds"
)


Orbit interpolation is now encapsulated in `effective_velocity.Estimator`.

The old `interp1d` position/velocity interpolation and the approximate
`V_r = sqrt(V_s V_g)` geometry are intentionally removed from this notebook.


## 5 - Raw-data correction, range compression, and Doppler centroid estimation


DCE được ước lượng từ chính dữ liệu range-compressed của chunk 13 và 14. Trước range compression, mean phức của từng chunk được trừ để hiệu chỉnh I/Q bias theo DAD §4.1 và §9.2.1. IPF không công bố scheduler của tập con raw-data analysis, nên notebook dùng toàn bộ từng chunk. Các record annotation bên dưới chỉ được dùng làm chuẩn đánh giá, không còn cấp Doppler Centroid cho bước focusing.


### 5.1 - I/Q bias correction (DAD §9.2.1)

Estimate the complex DC bias from the raw echo data before matched filtering.

In [ ]:
iq_bias = raw_data_correction.estimate_iq_bias(radar_data)


### 5.2 - Range compression (DAD §§6.1.1 and 6.2.2)

Correlate each echo line with the transmitted chirp replica and retain the valid convolution support.

In [ ]:
range_compressed, slant_range_time_vec_s = range_compression.compress(
    radar_data,
    raw_slant_range_time_vec_s,
    sample_rate_hz=range_sample_freq,
    pulse_start_frequency_hz=TXPSF,
    pulse_ramp_rate_hz_per_s=TXPRR,
    pulse_length_s=TXPL,
    iq_bias=iq_bias,
)
slant_range_vec_m = slant_range_time_vec_s * c / 2.0
len_range_line = range_compressed.shape[1]
del radar_data

print("Estimated I/Q bias:", iq_bias)
print("Range-compressed shape:", range_compressed.shape)
print("Valid range samples:", len_range_line)


In [ ]:
# Quick-look after range compression.
plt.figure(figsize=(12, 12))
plt.title("After Range Compression")
plt.imshow(
    np.abs(range_compressed[::20, ::20]),
    origin="lower",
    norm=colors.LogNorm()
)
plt.xlabel("Down Range (samples)")
plt.ylabel("Cross Range (samples)")
plt.show()


### 5.3 - Annotation records used as reference only

These Level-1 annotation polynomials are validation references; they are not inputs to focusing.

In [ ]:
DOPPLER_CENTROID_ANNOTATIONS = doppler_centroid_estimation.parse_annotation_records([
    {
        "azimuthTime": "2025-12-26T21:43:59.100490",
        "t0": 6.095910535477454e-03,
        "dataDcPolynomial": [4.152910e+01, 1.012491e+05, -4.252661e+08],
        "fineStart": "2025-12-26T21:43:57.297041",
        "fineStop": "2025-12-26T21:44:00.903940",
    },
    {
        "azimuthTime": "2025-12-26T21:44:14.095877",
        "t0": 6.095910535477454e-03,
        "dataDcPolynomial": [1.141131e+01, 1.275731e+04, 6.579813e+07],
        "fineStart": "2025-12-26T21:44:12.292428",
        "fineStop": "2025-12-26T21:44:15.899327",
    },
    {
        "azimuthTime": "2025-12-26T21:44:25.484967",
        "t0": 6.095910535477454e-03,
        "dataDcPolynomial": [3.263240e+01, -2.579159e+03, -1.314992e+08],
        "fineStart": "2025-12-26T21:44:23.681518",
        "fineStop": "2025-12-26T21:44:27.288417",
    },
])


### 5.4 - Doppler centroid input segments (DAD §5.6)

Load the adjacent acquisition chunk because the middle Doppler-estimation interval crosses the chunk boundary.

In [ ]:
ZERO_DOPPLER_MINUS_ACQ_TIME_S = 0.386295160  # Scene-specific inference.
DOPPLER_CENTROID_T0_S = DOPPLER_CENTROID_ANNOTATIONS[0]["t0"]

doppler_centroid_chunk = selected_chunk + 1
selection_14 = l0file.get_acquisition_chunk_metadata(doppler_centroid_chunk)
radar_data_14 = l0file.get_acquisition_chunk_data(doppler_centroid_chunk)
raw_range_count_14 = radar_data_14.shape[1]
range_start_time_14 = selection_14["SWST"].iloc[0] + suppressed_data_time
raw_slant_range_time_14 = (
    selection_14["Rank"].iloc[0] * selection_14["PRI"].iloc[0]
    + range_start_time_14
    + np.arange(raw_range_count_14) / range_sample_freq
)
packet_azimuth_times_14 = (
    selection_14["Coarse Time"].to_numpy(dtype=float)
    + selection_14["Fine Time"].to_numpy(dtype=float)
)


### 5.5 - Prepare the adjacent segment

Apply the same I/Q correction and range-compression settings before joining both segments on a common grid.

In [ ]:
iq_bias_14 = raw_data_correction.estimate_iq_bias(radar_data_14)
range_compressed_14, slant_range_time_14 = range_compression.compress(
    radar_data_14,
    raw_slant_range_time_14,
    sample_rate_hz=range_sample_freq,
    pulse_start_frequency_hz=TXPSF,
    pulse_ramp_rate_hz_per_s=TXPRR,
    pulse_length_s=TXPL,
    iq_bias=iq_bias_14,
)
del radar_data_14


### 5.6 - Fine and absolute Doppler centroid estimation (DAD §§5.2–5.5)

Estimate fine Doppler per range block, unwrap it, resolve the PRF ambiguity from geometry, and fit the range polynomial.

In [ ]:
doppler_centroid_segments = [
    doppler_centroid_estimation.Segment(
        range_compressed,
        slant_range_time_vec_s,
        packet_azimuth_times_s,
        name="chunk13",
    ),
    doppler_centroid_estimation.Segment(
        range_compressed_14,
        slant_range_time_14,
        packet_azimuth_times_14,
        name="chunk14",
    ),
]

scene_start_acq_s = packet_azimuth_times_s[0]
scene_stop_acq_s = packet_azimuth_times_14[-1] + PRI
doppler_centroid_estimator = doppler_centroid_estimation.Estimator.for_s6_research(
    prf_hz=az_sample_freq
)
doppler_centroid_estimates, doppler_centroid_scene = doppler_centroid_estimator.estimate_segments(
    doppler_centroid_segments,
    t0_s=DOPPLER_CENTROID_T0_S,
    slice_start_times_s=[scene_start_acq_s],
    last_slice_stop_time_s=scene_stop_acq_s,
    product_start_time_s=(
        scene_start_acq_s + ZERO_DOPPLER_MINUS_ACQ_TIME_S
    ),
    product_stop_time_s=(
        scene_stop_acq_s + ZERO_DOPPLER_MINUS_ACQ_TIME_S
    ),
    zero_dop_minus_acq_time_s=ZERO_DOPPLER_MINUS_ACQ_TIME_S,
    return_prepared_scene=True,
)

alignment_summary = pd.DataFrame(doppler_centroid_scene.alignment_summary())
display(alignment_summary)
print("Estimated Doppler centroid records:", len(doppler_centroid_estimates))


### 5.7 - Doppler centroid model for focusing

Expose the fitted Doppler centroid at each focusing-block center; annotation data is not used here.

In [ ]:
def doppler_centroid_for_block(block_center_index):
    return doppler_centroid_estimator.evaluate_for_line(
        doppler_centroid_estimates,
        line_index=block_center_index,
        azimuth_times_s=packet_azimuth_times_s,
        slant_range_times_s=slant_range_time_vec_s,
        azimuth_time_offset_s=ZERO_DOPPLER_MINUS_ACQ_TIME_S,
    )

# Estimated records no longer retain the large chunk-14 arrays.
del doppler_centroid_segments, doppler_centroid_scene, range_compressed_14


### 5.8 - Polynomial and accuracy comparison

Bảng hiển thị hệ số annotation/estimated, sai số thời gian, bias, MAE, RMSE, max error, coherence và RMS của phép fit CDCE. `ambiguity_adjusted_rmse_hz` chỉ là chẩn đoán sau khi loại bội nguyên PRF; focusing vẫn dùng trực tiếp đa thức tự ước lượng.


In [ ]:
dce_comparisons = doppler_centroid_estimation.compare_with_annotations(
    DOPPLER_CENTROID_ANNOTATIONS,
    doppler_centroid_estimates,
    slant_range_time_vec_s,
    prf_hz=az_sample_freq,
)

comparison_rows = []
for result in dce_comparisons:
    annotation_coeff = result["annotation_coefficients"]
    estimated_coeff = result["estimated_coefficients"]
    comparison_rows.append({
        "record": result["record"],
        "azimuth_time_error_ms": result["azimuth_time_error_ms"],
        "C0_annotation": annotation_coeff[0],
        "C0_estimated": estimated_coeff[0],
        "C0_error": estimated_coeff[0] - annotation_coeff[0],
        "C1_annotation": annotation_coeff[1],
        "C1_estimated": estimated_coeff[1],
        "C1_error": estimated_coeff[1] - annotation_coeff[1],
        "C2_annotation": annotation_coeff[2],
        "C2_estimated": estimated_coeff[2],
        "C2_error": estimated_coeff[2] - annotation_coeff[2],
        "bias_hz": result["bias_hz"],
        "mae_hz": result["mae_hz"],
        "rmse_hz": result["rmse_hz"],
        "max_abs_error_hz": result["max_abs_error_hz"],
        "integer_prf_adjustment_hz": result["integer_prf_adjustment_hz"],
        "ambiguity_adjusted_rmse_hz": result["ambiguity_adjusted_rmse_hz"],
        "CDCE_fit_rms_hz": result["fit_rms_hz"],
        "mean_coherence": result["mean_coherence"],
        "absolute_ambiguity_resolved": result["absolute_ambiguity_resolved"],
    })

dce_accuracy = pd.DataFrame(comparison_rows).set_index("record")
display(dce_accuracy)

fig, axes = plt.subplots(
    1, len(dce_comparisons), figsize=(18, 5), sharey=True
)
for axis, result in zip(np.atleast_1d(axes), dce_comparisons):
    range_time_ms = result["range_times_s"] * 1e3
    axis.plot(range_time_ms, result["annotation_hz"], label="Annotation record")
    axis.plot(range_time_ms, result["estimated_hz"], "--", label="Estimated CDCE")
    axis.set_title(
        f'DCE {result["record"]}: RMSE={result["rmse_hz"]:.3f} Hz'
    )
    axis.set_xlabel("Slant-range time [ms]")
    axis.grid(True, alpha=0.3)
axes[0].set_ylabel("Doppler centroid [Hz]")
axes[0].legend()
plt.tight_layout()
plt.show()

print(
    "Mean RMSE:", dce_accuracy["rmse_hz"].mean(), "Hz |",
    "Worst max abs error:", dce_accuracy["max_abs_error_hz"].max(), "Hz",
)
print(
    "Absolute ambiguity resolved:",
    dce_accuracy["absolute_ambiguity_resolved"].all(),
    "(False is expected without a geometry DC provider)",
)


## 6 - Effective velocity and focusing-block layout (DAD §§9.10–9.12)

Determine the azimuth matched-filter support from the orbit geometry, then size overlapping FFT blocks for continuous SLC assembly.

In [ ]:
# Sentinel-1 Stripmap S6 parameters from AUX_PP1.
AZIMUTH_PROCESSING_BANDWIDTH_HZ = 1398.0
FOCUS_FFT_LEN = 4096
EXTRA_AZIMUTH_PROCESSING_BLOCK_OVERLAP = 50
MAX_FDC_HZ = 100.0

slc_layout = slc_assembly.estimate_layout(
    len_az_line,
    slant_range_vec_m,
    packet_azimuth_times_s,
    doppler_centroid_for_block,
    vr_estimator,
    wavelength_m=wavelength_m,
    azimuth_sample_frequency_hz=az_sample_freq,
    processing_bandwidth_hz=AZIMUTH_PROCESSING_BANDWIDTH_HZ,
    fft_length=FOCUS_FFT_LEN,
    extra_overlap_samples=EXTRA_AZIMUTH_PROCESSING_BLOCK_OVERLAP,
)
MATCHED_FILTER_SUPPORT_SAMPLES = slc_layout.matched_filter_support_samples
FOCUS_OVERLAP_SAMPLES = slc_layout.overlap_samples
FOCUS_STEP = slc_layout.step_samples

print("Support probes:", slc_layout.support_probe_indices.tolist())
print("A probes:", list(slc_layout.support_probe_samples))
print("Configured/estimated A:", MATCHED_FILTER_SUPPORT_SAMPLES, "samples")
print("Total focus overlap:", FOCUS_OVERLAP_SAMPLES, "samples")
print("Focus step:", FOCUS_STEP, "samples")


## 7 - Secondary Range Compression (DAD §6.3.1)

Correct the range/azimuth coupling in the range-Doppler domain. `slc_assembly.assemble()` applies `src.apply()` to each azimuth block.


In [ ]:
SRC_SEGMENT_SAMPLES = 1024


## 8 - Range Cell Migration Correction (DAD §6.3.2)

Move each target's energy onto a constant range cell using fractional-delay sinc interpolation.


In [ ]:
RCMC_KERNEL_LENGTH = 16
RCMC_NUM_PHASES = 64


## 9 - Azimuth compression (DAD §6.3.4)

Build the azimuth matched filter from Doppler centroid and effective velocity. The block FFT, filter application, and inverse FFT run inside `slc_assembly.assemble()`.


In [ ]:
# Diagnostic: V_r is computed here, before focusing.
_vr_test_index = len_az_line // 2
_vr_test = vr_estimator.evaluate_block(
    block_center_time_s=packet_azimuth_times_s[_vr_test_index],
    slant_range_m=slant_range_vec_m,
    fdc_hz=doppler_centroid_for_block(_vr_test_index),
    azimuth_bandwidth_hz=AZIMUTH_PROCESSING_BANDWIDTH_HZ,
    n_control_points=9,
    range_polynomial_degree=2,
    return_diagnostics=True,
)
print(
    "V_r near/mid/far [m/s]:",
    _vr_test.vr_mps[0],
    _vr_test.vr_mps[len_range_line // 2],
    _vr_test.vr_mps[-1],
)
print(
    "Hyperbolic fit RMS range [m], max over control points:",
    max(point.fit_rms_m for point in _vr_test.control_points),
)
del _vr_test


## 10 - Focus overlapping blocks and assemble the SLC (DAD §9.12)

For every block, execute SRC → RCMC → azimuth compression, discard overlap support, and place the valid lines into the final SLC array.

In [ ]:
focused_image = slc_assembly.assemble(
    range_compressed,
    slant_range_vec_m,
    packet_azimuth_times_s,
    doppler_centroid_for_block,
    vr_estimator,
    slc_layout,
    wavelength_m=wavelength_m,
    speed_of_light_mps=c,
    azimuth_sample_period_s=az_sample_period,
    range_sample_period_s=range_sample_period,
    range_sample_frequency_hz=range_sample_freq,
    processing_bandwidth_hz=AZIMUTH_PROCESSING_BANDWIDTH_HZ,
    fft_length=FOCUS_FFT_LEN,
    src_segment_samples=SRC_SEGMENT_SAMPLES,
    rcmc_kernel_length=RCMC_KERNEL_LENGTH,
    rcmc_phases=RCMC_NUM_PHASES,
)
del range_compressed
print("Focused image shape:", focused_image.shape)


## 11 - Display focused image

In [ ]:
# Display scaling follows the normalized range matched filter.
tx_replica_sample_count = int(np.ceil(TXPL * range_sample_freq))
display_amplitude_scale = np.sqrt(tx_replica_sample_count)
valid_range_start = int(np.rint(
    (slant_range_time_vec_s[0] - raw_slant_range_time_vec_s[0])
    * range_sample_freq
))

plt.figure(figsize=(12, 12))
plt.title("Sentinel-1 Processed SAR Image")
plt.imshow(
    np.abs(focused_image[::20, ::20]),
    origin="lower",
    norm=colors.LogNorm(
        vmin=300 / display_amplitude_scale,
        vmax=10000 / display_amplitude_scale,
    ),
)
plt.xlabel("Down Range (samples)")
plt.ylabel("Cross Range (samples)")
plt.show()


In [ ]:
# Convert the Demo's raw-range window to the valid compressed-range grid.
plt.figure(figsize=(12, 12))
plt.title("Sentinel-1 Processed SAR Image - detail")
plt.imshow(
    np.abs(focused_image[
        9000:11000,
        6000 - valid_range_start:8000 - valid_range_start,
    ]),
    origin="lower",
    norm=colors.LogNorm(
        vmin=300 / display_amplitude_scale,
        vmax=10000 / display_amplitude_scale,
    ),
)
plt.xlabel("Down Range (samples)")
plt.ylabel("Cross Range (samples)")
plt.show()

elapsed = perf_counter() - notebook_started_at
print(f"Total runtime: {elapsed:.2f} s ({elapsed / 60:.2f} min)")


## 12 - Export amplitude TIFF for SNAP comparison

In [ ]:
# %pip install -q tifffile

# import tifffile

# tiff_path = "simple_focused_amplitude_raw_grid.tif"
# tiff = tifffile.memmap(
#     tiff_path,
#     shape=(len_az_line, raw_len_range_line),
#     dtype=np.float32,
#     bigtiff=True,
#     photometric="minisblack",
# )

# tiff[:, :valid_range_start] = np.nan
# tiff[:, valid_range_start + len_range_line:] = np.nan
# for a0 in range(0, len_az_line, RANGE_CHUNK):
#     a1 = min(a0 + RANGE_CHUNK, len_az_line)
#     tiff[a0:a1, valid_range_start:valid_range_start + len_range_line] = np.abs(focused_image[a0:a1])

# tiff.flush()
# del tiff
# print("Saved for SNAP:", tiff_path)
